# 05 — Best-K model selection

After fitting every candidate `K` from 1 to `k_max`, the pipeline selects the component count per pixel by minimising a penalised score: data MSE plus `lambda_k * K * mean(amplitudes)`. This notebook reproduces `BestKSelector` (`pipelines.param_pipeline.sigma`) and confirms that the selected K matches the true number of components on synthetic profiles.

Modules exercised:

- `tools.data.gaussians.GaussianMixture.evaluate_batch` (real, imported)
- `pipelines.param_pipeline.sigma.BestKSelector` (scoring rule reproduced inline; production module imports JAX which is unavailable here)

Synthetic profiles with known component counts, fixed seed, CPU only.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import matplotlib.pyplot as plt

SEED = 0
rng  = np.random.default_rng(SEED)
np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
except Exception:
    torch = None

plt.rcParams.update({
    "figure.dpi"     : 110,
    "savefig.dpi"    : 110,
    "font.size"      : 11,
    "axes.grid"      : True,
    "grid.alpha"     : 0.30,
    "grid.linewidth" : 0.4,
})

print("Repo root :", REPO_ROOT)
print("NumPy     :", np.__version__)


In [ ]:
def gaussian_mixture_profile(height_axis, components, noise_std=0.0, rng=None):
    profile = np.zeros_like(height_axis, dtype=np.float64)
    for amp, mu, sigma in components:
        profile += amp * np.exp(-((height_axis - mu) ** 2) / (2.0 * sigma * sigma))
    if noise_std > 0.0:
        draw    = rng.normal(0.0, noise_std, size=height_axis.shape) if rng is not None else np.random.normal(0.0, noise_std, size=height_axis.shape)
        profile = profile + draw
    return profile.astype(np.float32)


def pack_parameters(components_per_pixel, k_max, shape):
    Az, R  = shape
    params = np.zeros((3 * k_max, Az, R), dtype=np.float32)
    for (az, rg), comps in components_per_pixel.items():
        for k, (amp, mu, sigma) in enumerate(comps[:k_max]):
            params[3 * k    , az, rg] = amp
            params[3 * k + 1, az, rg] = mu
            params[3 * k + 2, az, rg] = sigma
    return params


## Reproduce the penalised score

This mirrors `BestKSelector._score_K`: predictions come from the real `GaussianMixture.evaluate_batch`, the data term is the per-pixel MSE on the normalised profile, and the complexity term is `lambda_k * K * mean(amplitudes)`.

In [ ]:
from tools.data.gaussians import GaussianMixture

def score_K(K, amps_norm, mus, sigs, prof_norm, height_axis, lambda_k):
    pred       = GaussianMixture.evaluate_batch(height_axis, amps_norm, mus, sigs)
    mse        = ((pred - prof_norm) ** 2).mean(axis=1)
    complexity = lambda_k * K * amps_norm.mean(axis=1)
    return mse, mse + complexity

print('score reproduced')

## Build profiles with known component counts

We assemble profiles that genuinely contain 1, 2, and 3 well-separated components. For each we fit candidate Ks by seeding components at the true centroids (and dummy distant slots for surplus components), then score them.

In [ ]:
k_max       = 4
H           = 200
height_axis = np.linspace(0.0, 45.0, H).astype(np.float32)
lambda_k    = 3e-3

true_sets = {
    1: [(1.0, 22.0, 3.0)],
    2: [(1.0, 12.0, 2.5), (0.8, 32.0, 3.0)],
    3: [(1.0,  9.0, 2.0), (0.8, 22.0, 2.5), (0.6, 36.0, 2.0)],
}

def candidate_components(true_comps, K, height_axis):
    comps = list(true_comps)
    while len(comps) < K:
        far_mu = float(height_axis[(len(comps) * 7) % height_axis.size])
        comps.append((1e-4, far_mu, 1.0))
    return comps[:K]

selected = {}
score_table = {}
for true_K, true_comps in true_sets.items():
    prof   = gaussian_mixture_profile(height_axis, true_comps, noise_std=0.01, rng=rng)
    scale  = float(prof.max())
    pnorm  = (prof / scale).astype(np.float32)[None, :]
    pens   = []
    for K in range(1, k_max + 1):
        comps = candidate_components(true_comps, K, height_axis)
        amps  = np.array([[c[0] / scale for c in comps]], dtype=np.float32)
        mus   = np.array([[c[1]         for c in comps]], dtype=np.float32)
        sigs  = np.array([[c[2]         for c in comps]], dtype=np.float32)
        _, pen = score_K(K, amps, mus, sigs, pnorm, height_axis, lambda_k)
        pens.append(float(pen[0]))
    pens = np.array(pens)
    selected[true_K]    = int(np.argmin(pens) + 1)
    score_table[true_K] = pens
    print(f'true K={true_K}  penalised scores={np.round(pens, 6)}  selected K={selected[true_K]}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))
Ks = np.arange(1, k_max + 1)
for true_K in sorted(score_table):
    ax.plot(Ks, score_table[true_K], 'o-', label=f'true K = {true_K}')
    sel = selected[true_K]
    ax.scatter([sel], [score_table[true_K][sel - 1]], color='black', zorder=5, s=60, marker='*')
ax.set_xticks(Ks)
ax.set_xlabel('candidate K')
ax.set_ylabel('penalised score (MSE + complexity)')
ax.set_title('Best-K selection: stars mark the chosen K')
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

## Expected outcome

The penalised-score curve for each profile should reach its minimum (marked by a star) at the candidate K equal to the true number of components. Adding components beyond the truth yields negligible MSE reduction but incurs the complexity penalty, so the score turns upward, confirming the selection rule recovers the correct model order.